## Analyze results good-quality mapping (batch sample, original counts, kl1)

In [ ]:
import scanpy as sc
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors

sys.path.append("../../../")
sys.path.append("../../utils/HLCA")
import mapqc
import data_cleanup

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

params

In [ ]:
study = "Sheppard_2020"
batch_variable = "sample"
testrun = False
use_old_counts = True
kl = 1.0
k = 500
distance_type = "edistance"

paths:

In [ ]:
if testrun:
    suffix = "_test"
    suffix_2 = ""
else:
    suffix = ""
    if use_old_counts == True:
        suffix_2 = "original_counts_"
    else:
        suffix_2 = "cellxgene_counts_"

path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_{study}_{suffix_2}batch_{batch_variable}_kl{kl}_latent{suffix}.h5ad"
path_counts = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_{study}_counts{suffix}.h5ad"
path_mapqc_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/mapqc_reproducibility/results/HLCA/{study}/{suffix_2}batch_{batch_variable}_kl{kl}{suffix}/mapqc"
path_figure_dir = (
    "/lustre/groups/ml01/code/lisa.sikkema/mapqc_figures/sheppard_good_mapping/"
)
path_table_dir = (
    "/lustre/groups/ml01/code/lisa.sikkema/mapqc_tables/sheppard_good_mapping/"
)

load relevant files:

In [ ]:
path_counts

In [ ]:
path_emb

In [ ]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)
adata = data_cleanup.add_mapping_info_to_counts_adata(adata_counts, adata_emb)

set gene symbols as adata.var row names (instead of IDs):

In [ ]:
adata.var["ensembl_id"] = adata.var.index
adata.var.index = [
    fn if not pd.isnull(fn) else gid
    for fn, gid in zip(adata.var.feature_name, adata.var.ensembl_id)
]

In [ ]:
results_healthy = mapqc.load(
    os.path.join(path_mapqc_results_dir, f"control_{distance_type}_k{k}.pkl")
)
results_disease = mapqc.load(
    os.path.join(path_mapqc_results_dir, f"case_{distance_type}_k{k}.pkl")
)

print stats:

In [ ]:
results_healthy.result_stats

In [ ]:
results_disease.result_stats

copy results to main adata:

In [ ]:
for stat in [
    "mapqc_score",
    "mapqc_score>2",
]:
    adata.obs[stat] = np.nan
    healthy_idc = [
        idx for idx in adata.obs.index if idx in results_healthy.adata.obs.index
    ]
    adata.obs.loc[healthy_idc, stat] = results_healthy.adata.obs.loc[healthy_idc, stat]
    disease_idc = [
        idx for idx in adata.obs.index if idx in results_disease.adata.obs.index
    ]
    adata.obs.loc[disease_idc, stat] = results_disease.adata.obs.loc[disease_idc, stat]

correct annotation hierarchy mistake in HLCA: pericytes are said to be part of the fibroblast supergroup, when they are actually a separate cell type

In [ ]:
for lev in [2, 3]:
    for ann_colname in [f"ann_level_{lev}", f"ann_level_{lev}_pred"]:
        adata.obs[ann_colname] = adata.obs[ann_colname].tolist()
        results_healthy.adata.obs[ann_colname] = results_healthy.adata.obs[
            ann_colname
        ].tolist()
        results_disease.adata.obs[ann_colname] = results_disease.adata.obs[
            ann_colname
        ].tolist()
        if "_pred" in ann_colname:
            ref_col = "ann_level_5_pred"
        else:
            ref_col = "ann_level_5"
        adata.obs.loc[
            adata.obs[ref_col] == "Pericytes",
            ann_colname,
        ] = "Pericytes"
        results_healthy.adata.obs.loc[
            results_healthy.adata.obs[ref_col] == "Pericytes",
            ann_colname,
        ] = "Pericytes"
        results_disease.adata.obs.loc[
            results_disease.adata.obs[ref_col] == "Pericytes",
            ann_colname,
        ] = "Pericytes"

## Plots:

In [ ]:
sc.set_figure_params(figsize=(7, 7))
s = 0.5

In [ ]:
FIGURES = dict()
TABLES = dict()

UMAP colored by reference, query control, query case

In [ ]:
# create variable splitting data in ref control, query control
adata.obs["r_or_q_ctr_or_case"] = [
    f"{r_q} {stat}" for r_q, stat in zip(adata.obs.r_or_q, adata.obs.status)
]
# clean up:
cleaning_dict = {
    "r control": "Reference (controls)",
    "q control": "Query controls",
    "q IPF": "Query IPF",
}
adata.obs.r_or_q_ctr_or_case = adata.obs.r_or_q_ctr_or_case.map(cleaning_dict)
results_healthy.adata.obs["r_or_q_ctr_or_case"] = adata.obs.loc[
    results_healthy.adata.obs.index, "r_or_q_ctr_or_case"
].copy()
results_disease.adata.obs["r_or_q_ctr_or_case"] = adata.obs.loc[
    results_disease.adata.obs.index, "r_or_q_ctr_or_case"
].copy()
palette_1 = {
    "Reference (controls)": "grey",
    "Query controls": "dodgerblue",
    "Query IPF": "midnightblue",
}
palette_boxplots = {  # need to use a different palette here, as with query dark blue we cannot see individual points on the boxes anymore
    "Reference (controls)": "grey",
    "Query controls": "#75bbfd",  # 8ab8fe",  # "lightblue",
    "Query IPF": matplotlib.colors.to_rgba(
        "#276ab3"
    ),  # 1e488f"),  # 015482"),  # 107ab0"),
}

In [ ]:
FIGURES["umap_r_or_q_ctr_or_case"] = sc.pl.umap(
    adata,
    color="r_or_q_ctr_or_case",
    frameon=False,
    ncols=1,
    palette=palette_1,
    show=False,
    s=s,
    return_fig=True,
)

UMAP colored by cell type annotation:  

In [ ]:
for lev in range(1, 6):
    adata.obs[f"ann_level_{lev}_merged"] = [
        ann if not pd.isnull(ann) else ann_pred
        for ann, ann_pred in zip(
            adata.obs[f"ann_level_{lev}"], adata.obs[f"ann_level_{lev}_pred"]
        )
    ]

In [ ]:
sc.pl.umap(adata, color="ann_level_2_merged", frameon=False, s=0.7)

In [ ]:
FIGURES["umap_anns"], ax = plt.subplots()
sc.pl.umap(adata, color="ann_level_3_merged", frameon=False, ax=ax, s=0.7)

UMAP colored by mapqc score controls:

In [ ]:
FIGURES["umap_mapqc_score"] = mapqc.pl.umap_mapqc_scores(
    results_healthy, return_fig=True
)

In [ ]:
FIGURES["umap_ctr_and_case_mapqc_score_binarized"], ax = plt.subplots(figsize=(6, 6))
sc.pl.umap(
    adata,
    color="mapqc_score>2",
    size=s,
    palette={
        ">2": "firebrick",
        "<=2": "navajowhite",
        "filtered out": "black",
        "not sampled": "cornflowerblue",
    },
    frameon=False,
    na_color="gray",
    ax=ax,
)

In [ ]:
FIGURES["umap_ctr_mapqc_score_binarized"] = mapqc.pl.umap_mapqc_scores(
    results_healthy,
    color_by="mapqc_score>2",
    return_fig=True,
    figsize=(6, 6),
    size=s * 2,
)

In [ ]:
FIGURES["umap_case_mapqc_score_binarized"] = mapqc.pl.umap_mapqc_scores(
    results_disease,
    color_by="mapqc_score>2",
    return_fig=True,
    figsize=(6, 6),
    size=s * 2,
)

In [ ]:
mapqc.pl.umap_filtering_info(results_disease)

Boxplot showing mapqc scores in controls per cell type:

In [ ]:
FIGURES["mapqc_per_ct_ctr_only"] = mapqc.pl.boxplot_mapqc_scores_per_category(
    results_healthy,
    box_category="ann_level_2_pred",
    figsize=(5, 4.5),
    point_size=0.3,
    color_category="donor_id",
    return_fig=True,
    min_cells_per_cat=150,
)

check number of outliers removed with vmin and vmax in figure below:

In [ ]:
(adata.obs.mapqc_score > 15).sum()

In [ ]:
(adata.obs.mapqc_score < -5).sum()

In [ ]:
FIGURES[
    "grouped_boxplot_lev2_ann_mapqc_score_cut_axes"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_2_pred",
    group_category="r_or_q_ctr_or_case",
    palette=palette_boxplots,
    figsize=(9, 5),
    vmax=15,
    vmin=-5,
    min_cells_per_box=100,
    return_fig=True,
)
plt.xlabel("Cell type")
plt.show()

In [ ]:
FIGURES[
    "grouped_boxplot_lev2_ann_mapqc_score"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_2_pred",
    group_category="r_or_q_ctr_or_case",
    palette=palette_boxplots,
    figsize=(9, 5),
    min_cells_per_box=100,
    return_fig=True,
)
plt.xlabel("Cell type")
plt.show()

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_3_pred",
    group_category="r_or_q_ctr_or_case",
    palette=palette_1,
    figsize=(12, 5),
    vmax=20,
    min_cells_per_box=50,
)

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_4_pred",
    group_category="r_or_q_ctr_or_case",
    palette=palette_1,
    figsize=(12, 5),
    vmax=20,
    min_cells_per_box=200,
)

### Analysis endothelial cells:

In [ ]:
FIGURES[
    "grouped_barplot_end_subtype_mapqc_scores"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_5_pred",
    group_category="r_or_q_ctr_or_case",
    box_groups_to_include=[
        "EC aerocyte capillary",
        "EC arterial",
        "EC general capillary",
        "EC venous pulmonary",
        "EC venous systemic",
    ],
    figsize=(5, 5),
    palette=palette_1,
    min_cells_per_box=50,
    return_fig=True,
)
plt.xlabel("Cell type")
plt.show()

subset to endothelial cells:

In [ ]:
adata_end = adata[adata.obs.ann_level_2_merged == "Blood vessels", :].copy()

re-run knn graph and umap

In [ ]:
sc.pp.neighbors(adata_end, n_neighbors=30, use_rep="X_scanvi_emb")
sc.tl.umap(adata_end)

In [ ]:
sc.tl.leiden(adata_end, key_added="leiden_end", resolution=0.3)

clean up EC annotations:

In [ ]:
adata_end.obs["ann_level_5_cleaned"] = data_cleanup.clean_up_df_column(
    adata_end.obs, column_name="ann_level_5_merged", min_n_cells_per_cat=50
)
palette_2 = {
    "EC aerocyte capillary": "#377eb8",
    "EC arterial": "#ff7f00",
    "EC general capillary": "#4daf4a",
    "EC venous pulmonary": "#f781bf",
    "EC venous systemic": "#a65628",
    "Other": "#a9a9a9",
}

In [ ]:
with plt.rc_context({"figure.figsize": (2, 2)}):
    sc.pl.umap(
        adata_end, color="ann_level_5_cleaned", palette=palette_2, frameon=False, s=0.5
    )

In [ ]:
sc.pl.umap(
    adata_end,
    color=[
        "r_or_q",
        "mapqc_score>2",
        "lung_condition",
        "leiden_end",
    ],
    wspace=0.3,
    frameon=False,
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    mapqc_results=results_healthy,
    box_category="ann_level_5_pred",
    groups_to_include=[
        "EC aerocyte capillary",
        "EC arterial",
        "EC general capillary",
        "EC venous pulmonary",
        "EC venous systemic",
    ],
    figsize=(2, 4),
    color_high_median_boxes_red=False,
    min_cells_per_cat=50,
)
plt.tight_layout()

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    mapqc_results=results_disease,
    box_category="ann_level_5_pred",
    groups_to_include=[
        "EC aerocyte capillary",
        "EC arterial",
        "EC general capillary",
        "EC venous pulmonary",
        "EC venous systemic",
    ],
    figsize=(2, 4),
    min_cells_per_cat=50,
    color_high_median_boxes_red=False,
)

show cluster make-up:

In [ ]:
FIGURES["EC_cluster_barplot"], ax = plt.subplots(figsize=(4, 2))
pd.crosstab(adata_end.obs.leiden_end, adata_end.obs.r_or_q_ctr_or_case).plot(
    kind="bar", stacked=True, color=palette_1, ax=ax
)
leg = plt.legend(loc=(1.01, 0.4), frameon=False, title="Cell source")
leg._legend_box.align = "left"
ax.spines[["right", "top"]].set_visible(False)
ax.set_ylabel("N cells")
ax.set_xlabel("Endothelial cell cluster")
plt.grid(False)
plt.show()

In [ ]:
FIGURES["EC_subtype_barplot"], ax = plt.subplots(figsize=(4, 2))
pd.crosstab(adata_end.obs.leiden_end, adata_end.obs.ann_level_5_cleaned).plot(
    kind="bar", stacked=True, ax=ax, color=palette_2
)
leg = plt.legend(loc=(1.01, 0), frameon=False, title="EC subtype")
leg._legend_box.align = "left"
ax.spines[["right", "top"]].set_visible(False)
plt.grid(False)
ax.set_ylabel("N cells")
ax.set_xlabel("Endothelial cell cluster")
plt.show()

marker plot:

In [ ]:
EC_markers = {
    "EC": ["SPARCL1", "AQP1", "RNASE1"],
    "EC arterial": ["DKK2", "IGFBP3"],
    "EC aerocyte capillary": ["SOSTDC1"],
    "EC general capillary": ["IL7R", "FCN3", "MT1M"],
    "EC venous systemic": ["TPD52L1", "ZNF385D", "OLFM1"],
    "EC venous pulmonary": [
        "CPE",
        "C7_ENSG00000112936",
        "MMRN1",
    ],  # gene with long name is actually C7 but somehow it is named with ensembl id here...
}

In [ ]:
adata_end.obs["ct_r_or_q"] = [
    f"{ct} ({r_or_q})"
    for ct, r_or_q in zip(adata_end.obs.ann_level_5_cleaned, adata_end.obs.r_or_q)
]

In [ ]:
sc.pl.dotplot(
    adata_end, var_names=EC_markers, groupby="ct_r_or_q", standard_scale="var"
)

### Smooth muscle analysis:

calculate percentage of cells with mapqc >2 for healthy and diseased in smooth muscle:

In [ ]:
results_healthy.adata.obs.loc[
    results_healthy.adata.obs.ann_level_2_pred == "Smooth muscle", "mapqc_score>2"
].value_counts() / sum(
    results_healthy.adata.obs.loc[
        results_healthy.adata.obs.ann_level_2_pred == "Smooth muscle", "mapqc_score>2"
    ].isin([">2", "<=2"])
) * 100

In [ ]:
results_disease.adata.obs.loc[
    results_disease.adata.obs.ann_level_2_pred == "Smooth muscle", "mapqc_score>2"
].value_counts() / sum(
    results_disease.adata.obs.loc[
        results_disease.adata.obs.ann_level_2_pred == "Smooth muscle", "mapqc_score>2"
    ].isin([">2", "<=2"])
) * 100

show mapqc scores but only for smooth muscle, coloring by donor:

In [ ]:
FIGURES[
    "grouped_boxplot_sm_mapqc_scores_per_donor"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_2_pred",
    group_category="donor_id",
    box_groups_to_include=["Smooth muscle"],
    figsize=(2, 3),
    min_cells_per_box=50,
    palette={
        "NML2": "#75bbfd",
        "NML1": "#75bbfd",
        "NML3": "#75bbfd",
        "IPF1": "#276ab3",
        "IPF2": "#276ab3",
        "IPF3": "#276ab3",
    },
    return_fig=True,
)

subset to mesenchyme except mesothelium:

In [ ]:
adata.obs.loc[adata.obs.ann_level_1 == "Stroma", "ann_level_2"].unique()

In [ ]:
adata_mes = adata[
    adata.obs.ann_level_2_merged.isin(
        ["Fibroblast lineage", "Pericytes", "Smooth muscle"]
    ),
    :,
].copy()

In [ ]:
adata_mes.shape

In [ ]:
sc.pp.filter_genes(adata_mes, min_cells=5)

In [ ]:
adata_mes.shape

(This clustering is used further down for the CTHRC1-high cluster analysis, so don't remove)

In [ ]:
sc.pp.neighbors(adata_mes, n_neighbors=30, use_rep="X_scanvi_emb")
sc.tl.umap(adata_mes)

In [ ]:
sc.tl.leiden(adata_mes, key_added="leiden_mes", resolution=0.3)

In [ ]:
sc.pl.umap(
    adata_mes,
    color=[
        "r_or_q",
        "mapqc_score>2",
        "lung_condition",
        "leiden_mes",
        "study",
        "ann_level_2_merged",
        "ann_level_3_merged",
        "CTHRC1",
    ],
    wspace=0.3,
    frameon=False,
)

##### gene-level analysis:

create obs column combining mapqc binary score and cell type annotation (both for adata_mes and for adata; the latter to check if the identified marker genes do not show e.g. ambient RNA patterns). Do the same for lung condition and cell type.

In [ ]:
# adata_mes (use level 3)
for lev in [3]:  # , 4]:
    obs_col_1 = f"ann_level_{lev}_mapqc_score"
    obs_col_2 = f"ann_level_{lev}_cond"
    adata_mes.obs[obs_col_1] = [
        f"{ct} ({stat})"
        for ct, stat in zip(
            adata_mes.obs[f"ann_level_{lev}_merged"],
            adata_mes.obs["mapqc_score>2"],  # .r_or_q_ctr_or_case
        )
    ]
    adata_mes.obs[obs_col_2] = [
        f"{ct} ({stat})"
        for ct, stat in zip(
            adata_mes.obs[f"ann_level_{lev}_merged"],
            adata_mes.obs["lung_condition"],  # .r_or_q_ctr_or_case
        )
    ]
    if lev in [2, 3]:
        min_n_cells = 200
    if lev == 4:
        min_n_cells = 50
    adata_mes.obs[obs_col_1] = data_cleanup.clean_up_df_column(
        adata_mes.obs, column_name=obs_col_1, min_n_cells_per_cat=min_n_cells
    )
    adata_mes.obs[obs_col_2] = data_cleanup.clean_up_df_column(
        adata_mes.obs, column_name=obs_col_2, min_n_cells_per_cat=min_n_cells
    )
    adata_mes.obs[obs_col_1] = adata_mes.obs[obs_col_1].str.replace("nan", "ref.")
    adata_mes.obs[obs_col_1] = adata_mes.obs[obs_col_1].str.replace(
        "<=2", "q., mapqc<2"
    )
    adata_mes.obs[obs_col_1] = adata_mes.obs[obs_col_1].str.replace(">2", "q., mapqc>2")
# adata
for lev in [2, 3, 4]:
    obs_col_1 = f"ann_level_{lev}_mapqc_score"
    obs_col_2 = f"ann_level_{lev}_cond"
    adata.obs[obs_col_1] = [
        f"{ct} ({stat})"
        for ct, stat in zip(
            adata.obs[f"ann_level_{lev}_merged"],
            adata.obs["mapqc_score>2"],
        )
    ]
    adata.obs[obs_col_2] = [
        f"{ct} ({stat})"
        for ct, stat in zip(
            adata.obs[f"ann_level_{lev}_merged"],
            adata.obs["lung_condition"],
        )
    ]
    if lev in [2]:
        min_n_cells = 200
    if lev in [3, 4]:
        min_n_cells = 50
    adata.obs[obs_col_1] = data_cleanup.clean_up_df_column(
        adata.obs, column_name=obs_col_1, min_n_cells_per_cat=min_n_cells
    )
    adata.obs[obs_col_2] = data_cleanup.clean_up_df_column(
        adata.obs, column_name=obs_col_2, min_n_cells_per_cat=min_n_cells
    )
    adata.obs[obs_col_1] = adata.obs[obs_col_1].str.replace("nan", "ref.")
    adata.obs[obs_col_1] = adata.obs[obs_col_1].str.replace("<=2", "q., mapqc<2")
    adata.obs[obs_col_1] = adata.obs[obs_col_1].str.replace(">2", "q., mapqc>2")

DEA on SM >2 vs SM <2 mapqc score:

In [ ]:
# TO DO HERE: PRINT GROUP SIZES

In [ ]:
mes_group_counts = adata_mes.obs.ann_level_3_mapqc_score.value_counts()
mes_group_counts.loc[
    [idx for idx in mes_group_counts.index if idx.startswith("Smooth muscle (q.,")]
]

In [ ]:
sc.tl.rank_genes_groups(
    adata_mes,
    groupby="ann_level_3_mapqc_score",
    groups=["Smooth muscle (q., mapqc>2)"],
    reference="Smooth muscle (q., mapqc<2)",
    key_added="SM high vs low",
)

In [ ]:
sm_deg_table = sc.get.rank_genes_groups_df(
    adata_mes, group="Smooth muscle (q., mapqc>2)", key="SM high vs low"
)

In [ ]:
sc.tl.filter_rank_genes_groups(
    adata_mes, key="SM high vs low", key_added="SM high vs low filt"
)  # , max_out_group_fraction=0.1)
sm_high_vs_low = sc.get.rank_genes_groups_df(
    adata_mes,
    group="Smooth muscle (q., mapqc>2)",
    key="SM high vs low filt",  # filtered"
).dropna(how="any")
sm_high_vs_low = sm_high_vs_low.loc[
    sm_high_vs_low.pvals_adj < 0.05, :
]  # only significant
sm_high_vs_low = sm_high_vs_low.loc[
    sm_high_vs_low.scores > 0, :
]  # only upregulated, for downregulated this filtering does not make sense

In [ ]:
sm_deg_table["passed_filtering"] = sm_deg_table.names.isin(sm_high_vs_low.names)
TABLES["smooth_muscle_query_high_vs_low_mapqc_score"] = sm_deg_table

check how many genes passed filtering:

In [ ]:
sm_high_vs_low.shape[0]

plot upregulated genes in SM >2 vs <2

In [ ]:
sc.pl.dotplot(
    adata_mes,
    var_names=sm_high_vs_low.names,
    groupby="ann_level_3_mapqc_score",
    standard_scale="var",
)

store markers that are cleanest, i.e. show lowest expression in low mapqc sm, highest in high mapqc sm, and that are as unique as possible based on visual inspection of the plot above:

In [ ]:
markers_sm_high_mapqc_manual = [
    "IGFBP2",
    "RGS5",
    "STEAP4",
    "FAM162B",
    "TCIM",
    "CCL21",
    "ARHGAP15",
    "CD36",
    "PTGER1",
]
markers_sm_high_mapqc_top10 = sm_high_vs_low.names[:10].tolist()

and plot same genes as in plot above, now subdividing cell types by condition. In that way we can see if markers are indeed mostly (or only) expressed in IPF.

In [ ]:
sc.pl.dotplot(
    adata_mes,
    var_names=sm_high_vs_low.names,
    groupby="ann_level_3_cond",
    standard_scale="var",
)

now plot subset of "high quality" differentially expressed genes defined above. Plot per mesenchyme cell type, splitting by condition

In [ ]:
sc.pl.dotplot(
    adata_mes,
    var_names=markers_sm_high_mapqc_manual,
    groupby="ann_level_3_cond",
    standard_scale="var",
)

plot genes for smooth muscle only:

In [ ]:
sc.pl.dotplot(
    adata_mes[adata_mes.obs.ann_level_3_merged == "Smooth muscle", :],
    var_names=markers_sm_high_mapqc_manual,
    groupby="ann_level_3_mapqc_score",
    standard_scale="var",
    vmin=0,
)

create .obs column specifying whether a cell came from teh reference or query, and if it came from the query, if it had a mapqc score above 2:

In [ ]:
adata_mes.obs["mapqc_r_or_q"] = [
    "Ref." if r_or_q == "r" else f"mapQC score{mapqc} (q.)"
    for r_or_q, mapqc in zip(adata_mes.obs.r_or_q, adata_mes.obs["mapqc_score>2"])
]
adata_mes.obs["mapqc_r_or_q"] = data_cleanup.clean_up_df_column(
    adata_mes.obs, "mapqc_r_or_q", min_n_cells_per_cat=100
)
# order categories for figure:
adata_mes.obs.mapqc_r_or_q = pd.Categorical(
    adata_mes.obs.mapqc_r_or_q,
    categories=[
        "Ref.",
        "mapQC score<=2 (q.)",
        "mapQC score>2 (q.)",
        "mapQC scorenot sampled (q.)",
        "Other",
    ],
    ordered=True,
)

In [ ]:
adata_mes.obs.mapqc_r_or_q.value_counts()

Now plot markers for smooth muscle subsets only:

In [ ]:
sc.pl.dotplot(
    adata_mes[
        (adata_mes.obs.ann_level_3_merged == "Smooth muscle")
        & adata_mes.obs.mapqc_r_or_q.isin(
            [
                "Ref.",
                "mapQC score<=2 (q.)",
                "mapQC score>2 (q.)",
            ]
        ),
        :,
    ],
    var_names=markers_sm_high_mapqc_manual,
    groupby="mapqc_r_or_q",
    # standard_scale="var",
    vmin=0,
)

check results in whole adata, to check for doublets, ambient RNA patterns etc.:

In [ ]:
markers_mes = {
    "pericyte": ["LAMC3", "HIGD1B", "PTN"],
    "smooth muscle": [
        "MYH11",
        "TINAGL1",
        "PLN",
        "ACTA2",
        "TAGLN",
    ],  # last two from Martijn and Elo, first three from HLCA
    "sm bad vs good mapping": markers_sm_high_mapqc_manual,
    "fetal pericyte paper 1 and DEG": ["MAP3K7CL"],
    "fetal pericyte 1": ["GPC3", "SPON2", "TGFB1"],
    "fetal pericyte 2": [
        "NET1",
        "PLN",
        "ADIRF",
        "FRZB",
        "NUDT4",
    ],  # a few of these are re-expressed in IPF-specific pericytes!
    # some doublet checks:
    "AT0 or alveolar epithelium": [
        "SLC39A8",  # from here on: AT0 and alveolar epithelium markers
        "SPOCK2",
        "MFSD2A",
        "BIRC5",
        "SFTPB",
    ],
    "endothelium": ["CDH5"],
    "immune": ["PTPRC"],
    "epithelium": ["EPCAM"],
}  # check other alveolar epithelium markers to check for doublets or misannotations

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=markers_mes,
    groupby="ann_level_2_mapqc_score",
    standard_scale="var",
)

Conclusion: there is some pericyte-smooth muscle intermediate state, with cells expressing both a subset of pericyte markers (HIGD1B, PTN, rarely expressed in SM), and genes normally only (in the stroma) expressed in SM (RGS16, FRZB, MAP3K7CL, STEAP4 and more, these are outcome of DEA of Pericytes mapqc>2 versus pericyes ref, with ref being consistent with <2 query). Does this group of cells also show a unique marker --> ARHGAP15

To show: whereas IPF pericytes show normal expression of some SM markers (PLN is low) and of some pericyte markers (HIG1DB high), they show increased expression of other SM markers or SM specific genes (RGS16, FRZB, MAP3K7CL, STEAP4), and decreased expression of some pericyte markers (LAMC3 low).<br>
Similarly, whereas IPF smooth muscle show normal or higher expression of SM markers (RGS16, FRZB, STEAP4, MAP3K7CL extra high, PLN a little bit lower), they show increased expression of pericyte markers (mostly PTN).<br>
Check 1: is this due to mis-annotations? Then there would be no or few double-positives, but in fact the number of DPs is very high in pericytes (see e.g. HIG1DB-RGS16 or HIG1DB-FRZB), and also relatively high in SM (see PTN_PLN or PTN_FRZB).<br>
Check 2: are these not doublets? This can only be proven by gene more highly expressed in this double-positive group specifically than in any of the other two groups. This is the case for ARHGAP15, LZTS1, CTSC, HOPX, for the pericyte double positives, and increased expression compared to both SM and pericytes for MAP3K7CL (output of DEA strict filtering for both SM and pericyte).

In [ ]:
groups_of_interest = [
    g
    for g in adata_mes.obs.ann_level_3_mapqc_score.unique()
    if (g.startswith("Peri") or g.startswith("Smooth"))
]

In [ ]:
double_positive_df = pd.DataFrame(index=sorted(groups_of_interest))
single_genes_not_for_dp = ["ARHGAP15", "LZTS1", "CTSC", "HOPX", "MAP3K7CL"]
single_pos_calculated = []
for peri_gene in ["HIGD1B", "PTN"]:
    for sm_gene in ["RGS16", "FRZB", "MAP3K7CL", "STEAP4", "PLN"]:
        gene_df = pd.DataFrame(index=adata_mes.obs.index)
        gene_df["ann"] = adata_mes.obs.ann_level_3_mapqc_score
        gene_df[peri_gene] = adata_mes[:, peri_gene].X.toarray()
        gene_df[sm_gene] = adata_mes[:, sm_gene].X.toarray()
        for single_gene in [peri_gene, sm_gene] + single_genes_not_for_dp:
            if single_gene not in single_pos_calculated:
                if single_gene not in gene_df.columns:
                    gene_df[single_gene] = adata_mes[:, single_gene].X.toarray()
                # calculate positive fraction for single gene:
                gene_df["sp"] = gene_df[single_gene] > 0
                sp_frac = gene_df.groupby("ann")["sp"].apply(lambda x: (x > 0).mean())
                double_positive_df[single_gene] = round(
                    sp_frac.loc[double_positive_df.index], 3
                )
                single_pos_calculated.append(single_gene)
        # calculate positive fraction for single sm gene
        gene_df["dp"] = (gene_df[peri_gene] > 0) & (gene_df[sm_gene] > 0)
        adata_mes.obs[f"dp_{peri_gene}_{sm_gene}"] = gene_df["dp"]
        dp_frac = gene_df.groupby("ann")["dp"].apply(lambda x: (x > 0).mean())
        double_positive_df[f"{peri_gene}_{sm_gene}"] = round(
            dp_frac.loc[double_positive_df.index], 3
        )

In [ ]:
# also calculate mean total counts per group:
adata_mes.obs.total_counts = adata_mes.X.sum(axis=1)
group_mean_total_count = adata_mes.obs.groupby("ann_level_3_mapqc_score").agg(
    {"total_counts": "mean"}
)
double_positive_df["total_count_norm"] = (
    group_mean_total_count.loc[double_positive_df.index] / group_mean_total_count.max()
)

In [ ]:
import seaborn as sns

In [ ]:
sns.heatmap(
    double_positive_df.loc[
        [
            "Pericytes (q., mapqc<2)",
            "Pericytes (q., mapqc>2)",
            "Pericytes (ref.)",
            "Smooth muscle (q., mapqc<2)",
            "Smooth muscle (q., mapqc>2)",
            "Smooth muscle (ref.)",
        ],
        ["total_count_norm"]
        # + sorted(
        #     [
        #         col
        #         for col in double_positive_df.columns
        #         if ("_" in col and col != "total_count_norm")
        #     ]
        + [
            "PTN_FRZB",
            "PTN_MAP3K7CL",
        ]
        + single_genes_not_for_dp,
    ],
    cmap="Reds",
)

check results in umap of mesenchyme:

In [ ]:
for ct in [
    "smooth muscle",
    "pericyte",
    "fetal pericyte paper 1 and DEG",
    "fetal pericyte 2",
]:
    print(ct.upper())
    sc.pl.umap(adata_mes, color=markers_mes[ct], frameon=False, ncols=5)
    print("\n\n\n")
# add handpicked genes for >2 pericytes:
sc.pl.umap(adata_mes, color=markers_mes[ct], frameon=False, ncols=5)

In [ ]:
sc.pl.umap(
    adata_mes,
    color=[
        "r_or_q",
        "mapqc_score>2",
        "lung_condition",
        "leiden_mes",
        "study",
        "ann_level_3_merged",
        "ann_level_4_merged",
    ]
    + [col for col in adata_mes.obs.columns if col.startswith("dp")],
    frameon=False,
    ncols=5,
)

In [ ]:
mes_markers_intermediate = {
    "SM marker low in all pericyte": ["PLN"],
    "Pericyte marker high in all pericyte": ["HIGD1B", "PTN"],  # I need more of these!
    "SM marker high in IPF pericyte": ["RGS16", "FRZB", "MAP3K7CL", "STEAP4"],
    "Pericyte marker low in IPF pericyte": ["LAMC3"],
    "SM marker high in all SM": ["RGS16", "FRZB", "STEAP4", "MAP3K7CL"],
    # "Pericyte marker higher in IPF SM": ["PTN"],
    "Pericyte marker high in IPF peri only": ["ARHGAP15", "LZTS1", "CTSC", "HOPX"],
    "Marker extra high in IPF peri and SM": ["MAP3K7CL"],
    "Pericyte marker low in IPF peri": [
        "TBX3",
        "CPM",
        "EZR",
        "RGN",
        "PAG1",
        "OTULINL",
        "PAG1",
        "KCNK3",
    ],  # cut this down further
    "Fibroblast IPF marker from paper": ["CTHRC1"],
}

In [ ]:
sc.pl.dotplot(
    adata_mes,
    var_names=mes_markers_intermediate,
    groupby="ann_level_3_mapqc_score",
    standard_scale="var",
)

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=mes_markers_intermediate,
    groupby="ann_level_2_mapqc_score",
    # standard_scale="var",
)

Paper text draft: Gene expression patterns suggest that pericytes with high mapQC scores (>2), as observed in large numbers in samples from IPF patients, in fact constitute an intermediate cell state between healthy pericytes and smooth muscle cells. These pericytes show expression of genes normally expressed in smooth muscle and not in pericytes (FRZB, MAP3K7CL), while also consistently expressing (>80% of cells positive) a pericyte-specific marker not expressed in smooth muscle (PTN) [removed HIG1DB from here as it is less clean]. Notably, joint expression of these genes is rarely observed in healthy lung pericytes or smooth muscle cells [fig. heatmap], but has been observed in a subset of pericytes of the developing lung [vascular SMC2]. While not fully recapitulating transcriptomic patterns of this embryonic counterpart, both cell types also [Removing: These pericytes moreover express RGS16, normally primarily expressed in activated or stressed smooth muscle] lack  expression of genes highly expressed in healthy, adult pericytes (RGN, PAG1, TBX3, LAMC3). [removed EZR as not recapitulated in embryonic ct, and also not as clean in HLCA as the others). [removing this as not recapitulated in embryonic state, need to keep it short: or in both smooth muscle and pericytes (EZR, MYH11)].

In [ ]:
markers_paper_figure = {
    "Pericyte,\nall states": ["PTN", "HIGD1B"],
    "Pericyte,\nlacking in IPF state": ["LAMC3", "RGN", "PAG1", "TBX3"],
    "SM, also in IPF pericytes": ["FRZB", "MAP3K7CL"],
    "SM, not in pericytes": ["PLN"],
}

In [ ]:
groups_to_show_hlca = [
    gr
    for gr in adata_mes.obs.ann_level_3_mapqc_score.unique()
    if (("smooth muscle (" in gr.lower()) or ("peri" in gr.lower()))
]
sc.pl.dotplot(
    adata_mes[adata_mes.obs.ann_level_3_mapqc_score.isin(groups_to_show_hlca), :],
    var_names=markers_paper_figure,
    groupby="ann_level_3_mapqc_score",
    standard_scale="var",
    use_raw=False,
)

In [ ]:
adata_mes.obs.ann_level_3_pred.value_counts()

In [ ]:
data = pd.DataFrame()
data["x"] = adata_mes.obs.loc[
    adata_mes.obs.ann_level_3_pred == "Pericytes", "mapqc_score"
]
data["y"] = adata_mes[adata_mes.obs.ann_level_3_pred == "Pericytes", "FRZB"].X.toarray()
data.dropna(how="any", inplace=True)
data["x"] = data["x"].astype(float)
data["y"] = data["y"].astype(float)
# Create bins for x (you can adjust the number of bins)
# Specify exact bin ranges
bin_edges = np.arange(-2, 8, 1)  # np.floor(np.min(x)), np.ceil(np.max(x)), 0.5)

# Create bins and labels
data["x_binned"] = pd.cut(data["x"], bins=bin_edges)

# Create labels
labels = [
    f"[{bin_edges[i]:.1f}, {bin_edges[i+1]:.1f}]" for i in range(len(bin_edges) - 1)
]

# Create the plot
plt.figure(figsize=(14, 6))
sns.violinplot(x="x_binned", y="y", data=data)

# Customize the plot
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.xlabel("X bins")
plt.ylabel("Y values")
plt.title("Distribution of Y values across X bins")

plt.tight_layout()
plt.show()

In [ ]:
adata_developmental = sc.read_h5ad(
    "/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/developing_lung/he_et_al_fibroblasts_sm.h5ad"
)

In [ ]:
adata_developmental.var["ensembl_id"] = adata_developmental.var.index
adata_developmental.var.index = adata_developmental.var.gene_symbols

In [ ]:
groups_to_show_dev = [
    gr
    for gr in adata_developmental.obs.new_celltype.unique()
    if (("sm" in gr.lower()) or ("peri" in gr.lower()))
]
sc.pl.dotplot(
    adata_developmental[
        adata_developmental.obs.new_celltype.isin(groups_to_show_dev), :
    ],
    var_names=markers_paper_figure,
    groupby="new_celltype",
    standard_scale="var",
    use_raw=False,
)

check markers from madissoon et al spatial lung paper (Outcome: healthy pericytes are all pulmonary pericytes, IPF ones (mapqc>2) are likely also but altered, don't show signs of any alternative peri subtypes.)

In [ ]:
madissoon_markers = {
    "ASM": ["HS6ST3", "GPM6A", "DOCK3", "COL4A6", "HPSE2"],
    "IR-Ven-Peri": ["FGF7"],
    "Peri-pulm": ["KCNK3", "MEST", "LAMC3", "PTN"],
    "Peri-syst": ["FRMD3", "SLC38A11"],
    "SM-pulm": ["TNNT2"],  # or for both SM groups: ['TGM2']
    "SM-Art_syst": ["SORBS2", "SYNM", "TGM2"],  # no unique markers...
}

In [ ]:
sc.pl.dotplot(
    adata_mes,
    var_names=madissoon_markers,
    groupby="ann_level_3_mapqc_score",
    standard_scale="var",
)

### Alveolar fibroblast sanity check:

In [ ]:
sc.pl.umap(adata, color=["CTHRC1"], frameon=False, vmax="p99.5")

In [ ]:
adata.obs["leiden_mes"] = np.nan
adata.obs.loc[adata_mes.obs.index, "leiden_mes"] = adata_mes.obs.leiden_mes

In [ ]:
sc.pl.umap(adata, color="leiden_mes", frameon=False, groups=["3"])

plot CTHRC1 expression for samples in leiden_mes cluster 3:

In [ ]:
adata_mes_leiden_3 = adata_mes[adata_mes.obs.leiden_mes == "3", :].copy()
adata_mes_leiden_3.obs["sample_clean"] = data_cleanup.clean_up_df_column(
    adata_mes_leiden_3.obs, "sample", min_n_cells_per_cat=10
)
adata_mes_leiden_3.obs["donor_clean"] = data_cleanup.clean_up_df_column(
    adata_mes_leiden_3.obs, "donor_id", min_n_cells_per_cat=10
)
adata_mes_leiden_3.obs["sample_filtered_st_cond"] = [
    "Other" if s == "Other" else f"{s} ({st}, {cond})"
    for s, st, cond in zip(
        adata_mes_leiden_3.obs.sample_clean,
        adata_mes_leiden_3.obs.study,
        adata_mes_leiden_3.obs.lung_condition,
    )
]
adata_mes_leiden_3.obs["donor_filtered_st_cond"] = [
    "Other" if d == "Other" else f"{d} ({st}, {cond})"
    for d, st, cond in zip(
        adata_mes_leiden_3.obs.donor_clean,
        adata_mes_leiden_3.obs.study,
        adata_mes_leiden_3.obs.lung_condition,
    )
]
adata_mes_leiden_3.obs["CTHRC1"] = adata_mes_leiden_3[:, "CTHRC1"].X.toarray()

In [ ]:
adata_mes.obs.r_or_q_ctr_or_case.value_counts()

In [ ]:
FIGURES["boxplot_CTHRC1_expr_per_donor"], ax = plt.subplots(figsize=(12, 4))
donor_info = (
    adata_mes_leiden_3.obs.groupby("donor_filtered_st_cond")
    .agg(
        {
            "CTHRC1": "mean",
            "study": "first",
            "lung_condition": "first",
            "donor_filtered_st_cond": "count",
            "donor_id": "first",
            "r_or_q_ctr_or_case": "first",
        }
    )
    .sort_values(by="CTHRC1", ascending=False)
)
# remove other category
donor_order = donor_info.loc[donor_info.index != "Other", :].copy()
donor_order = donor_info.index
# simplify condition (change healthy, tumor adjacent to healthy):
donor_info.loc[
    donor_info.lung_condition == "Healthy (tumor adjacent)", "lung_condition"
] = "Healthy"
x_tick_labels = [
    f"{cond} ({st}, n={n})"
    for cond, st, n in zip(
        donor_info.lung_condition, donor_info.study, donor_info.donor_filtered_st_cond
    )
]
palette_2 = {
    d: palette_boxplots[status]
    for d, status in zip(donor_info.index, donor_info.r_or_q_ctr_or_case)
}
sns.boxplot(
    adata_mes_leiden_3.obs,
    x="donor_filtered_st_cond",
    y="CTHRC1",
    order=donor_order,
    ax=ax,
    palette=palette_2,
    saturation=0.5,
    fliersize=0,
)
sns.stripplot(
    adata_mes_leiden_3.obs,
    x="donor_filtered_st_cond",
    y="CTHRC1",
    order=donor_order,
    ax=ax,
    color="black",
    s=1,
)
ax.tick_params(axis="x", rotation=90)
ax.set_xticks(ticks=range(len(x_tick_labels)), labels=x_tick_labels)
ax.set_xlabel("Donor lung condition (study, n cells)")
plt.show()

In [ ]:
min_n_cells = 10
df_leiden_3_donor = adata_mes_leiden_3.obs.groupby("donor_id").agg(
    {
        "study": "first",
        "CTHRC1": "mean",
        "lung_condition": "first",
        "r_or_q_ctr_or_case": "first",
        "donor_id": "count",
    }
)
# keep only donors with at least min_n_cells
print("Shape before filter:", df_leiden_3_donor.shape)
df_leiden_3_donor = df_leiden_3_donor.loc[
    df_leiden_3_donor.donor_id > min_n_cells, :
].copy()
print("Shape after filter:", df_leiden_3_donor.shape)

In [ ]:
FIGURES["boxplot_CTHRC1_expr_ref_q_ctr_case"], ax = plt.subplots(figsize=(2.5, 5))
sns.boxplot(
    df_leiden_3_donor,
    x="r_or_q_ctr_or_case",
    y="CTHRC1",
    order=["Reference (controls)", "Query IPF", "Query controls"],
    ax=ax,
    palette=palette_boxplots,
    saturation=1,
    fliersize=0,
)
sns.stripplot(
    df_leiden_3_donor,
    x="r_or_q_ctr_or_case",
    y="CTHRC1",
    order=["Reference (controls)", "Query IPF", "Query controls"],
    ax=ax,
    color="black",
    s=3,
)
plt.grid(False)
plt.xticks(fontsize=12, rotation=90)
plt.yticks(fontsize=12, rotation=90)
ax.set_ylabel("Mean donor\nCTHRC1 expression", fontsize=12)
ax.set_xlabel("Reference or query,\nlung condition", fontsize=12)
plt.tight_layout()
plt.show()

### Macrophage sanity check:

In [ ]:
FIGURES[
    "grouped_boxplot_mac_subtype_mapqc_scores"
] = mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    mapqc_results_ctr=results_healthy,
    mapqc_results_case=results_disease,
    box_category="ann_level_4_pred",
    group_category="r_or_q_ctr_or_case",
    box_groups_to_include=["Alveolar macrophages", "Interstitial macrophages"],
    figsize=(3, 5),
    palette=palette_boxplots,
    min_cells_per_box=50,
    pointsize=0.4,
    return_fig=True,
)
plt.xlabel("Cell type")
plt.show()

In [ ]:
adata_macs = adata[adata.obs.ann_level_3_merged == "Macrophages", :].copy()

In [ ]:
# adata
for lev in [4]:
    obs_col = f"ann_level_{lev}_mapqc_score_clean"
    # first simplify ct annotation (set small groups to other)
    adata_macs.obs[f"ann_level_{lev}_merged_clean"] = data_cleanup.clean_up_df_column(
        adata_macs.obs, column_name=f"ann_level_{lev}_merged", min_n_cells_per_cat=100
    )
    adata_macs.obs[obs_col] = [
        f"{ct} ({stat})"
        for ct, stat in zip(
            adata_macs.obs[f"ann_level_{lev}_merged_clean"],
            adata_macs.obs["mapqc_score>2"],
        )
    ]
    adata_macs.obs[obs_col] = data_cleanup.clean_up_df_column(
        adata_macs.obs, column_name=obs_col, min_n_cells_per_cat=50
    )
    adata_macs.obs[obs_col] = adata_macs.obs[obs_col].str.replace("nan", "ref.")
    adata_macs.obs[obs_col] = adata_macs.obs[obs_col].str.replace("<=2", "q., mapqc<2")
    adata_macs.obs[obs_col] = adata_macs.obs[obs_col].str.replace(">2", "q., mapqc>2")

In [ ]:
adata_macs.obs.ann_level_4_mapqc_score_clean.value_counts()

In [ ]:
adata_macs.obs["mapqc_r_or_q"] = [
    "Ref." if r_or_q == "r" else f"mapQC score{mapqc} (q.)"
    for r_or_q, mapqc in zip(adata_macs.obs.r_or_q, adata_macs.obs["mapqc_score>2"])
]
adata_macs.obs["mapqc_r_or_q"] = data_cleanup.clean_up_df_column(
    adata_macs.obs, "mapqc_r_or_q", min_n_cells_per_cat=100
)
adata_macs.obs.mapqc_r_or_q = pd.Categorical(
    adata_macs.obs.mapqc_r_or_q,
    categories=[
        "Ref.",
        "mapQC score<=2 (q.)",
        "mapQC score>2 (q.)",
        "mapQC scorenot sampled (q.)",
        "Other",
    ],
    ordered=True,
)

Order categories for plot:

In [ ]:
adata_macs.obs.mapqc_r_or_q = pd.Categorical(
    adata_macs.obs.mapqc_r_or_q,
    categories=[
        "Ref.",
        "mapQC score<=2 (q.)",
        "mapQC score>2 (q.)",
        "mapQC scorenot sampled (q.)",
        "Other",
    ],
    ordered=True,
)

In [ ]:
sc.pl.dotplot(
    adata_macs[
        (adata_macs.obs.ann_level_4_merged == "Alveolar macrophages")
        & (
            adata_macs.obs.mapqc_r_or_q.isin(
                ["Ref.", "mapQC score<=2 (q.)", "mapQC score>2 (q.)"]
            )
        ),
        :,
    ],
    var_names={
        "Alveolar macrophage": [
            "MARCO",
        ],  # cd11c = itgax siglec1=cd169
        # "Interstitial macrophage markers": ["F13A1", "FOLR2"],
        "Down in IPF": ["FABP4", "INHBA"],
        "Up in IPF": [
            "SPP1",
        ],
    },
    groupby="mapqc_r_or_q",
    # standard_scale="var",
)

## Store figures and tables:

In [ ]:
FIGURES

In [ ]:
for figname, fig in FIGURES.items():
    # if "umap" not in figname:  # TEMP FOR FASTER SAVING, ALREADY HAVE ALL UMAPS
    print("Saving", figname)
    fig.savefig(
        os.path.join(path_figure_dir, f"{figname}.png"),
        bbox_inches="tight",
        dpi=140,
        transparent=True,
    )

In [ ]:
TABLES.keys()

In [ ]:
TABLES["smooth_muscle_query_high_vs_low_mapqc_score"].to_csv(
    os.path.join(path_table_dir, "smooth_muscle_query_high_vs_low_mapqc_score.csv"),
    index=False,
)